# 🤖 LLM Fundamentals for Business Managers
### Essential Basics You Should Know (15-20 minutes)

---
## ⚙️ Setup - Run This First

In [ ]:

# Install required packages
!pip install litellm tiktoken python-dotenv ipython -q

import os
from dotenv import load_dotenv
from litellm import completion
import tiktoken
from IPython.display import Markdown, display
load_dotenv()

print("✅ Setup complete. You are ready to go!")

---
## 1. What Are LLMs Really Doing?

Large Language Models are **advanced next-word predictors**. 

They don't truly understand — they predict what word is most likely to come next based on patterns seen during training.

---
## 2. Tokens — The Real Unit of Cost

In [ ]:
enc = tiktoken.encoding_for_model("gpt-4o")

examples = ["Hello, how are you?", "你好吗?", "Apa khabar?", "Hallo, wie geht's dir?"]

for text in examples:
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"Words: {words} | Tokens: {tokens} | Ratio: {(tokens/words) if words > 0 else 0:.1f} tokens per word")

**Business Insight:** You are billed per token, not per word.

💡 **Cost Context:** Summarizing a 10-page document (~5,000 tokens) with gpt-4o-mini costs roughly $0.0005 less than a penny.

---
## 3. Next Word Prediction

In [ ]:
def show_probability(prompt):
    resp = completion(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        logprobs=True,
        top_logprobs=5
    )
    print(f"Prompt: '{prompt}'")
    for item in resp.choices[0].logprobs.content[0].top_logprobs:
        prob = round((2.71828 ** item.logprob) * 100, 1)
        print(f"  {item.token:<15} {prob:6.1f}%")

show_probability("The capital of France is")
show_probability("A good way to reduce IT support costs is")
show_probability("A man was running so fast because")

---
## 🆕 3.5 Temperature & The Hallucination Warning

### Temperature: Predictable vs Creative

In [3]:

CREATIVE_PROMPT = "Write one short, catchy tagline for an AI-powered IT helpdesk. Max 10 words."

for label, temp in [("temp=0.0", 0.0), ("temp=1.0", 1.0)]:
    seen = set()
    print(f"   {label} (5 runs):")
    for i in range(5):
        r = completion(
            model="gpt-4o",
            messages=[{"role": "user", "content": CREATIVE_PROMPT}],
            temperature=temp, max_tokens=30
        )
        ans = r.choices[0].message.content.strip()
        seen.add(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique answers: {len(seen)}/5")
    print()

print("🔑 RULE: temp=0 for decisions/routing. temp=0.7–1.0 for drafting/brainstorming.")
print("   In any automation based pipeline: ALWAYS use temp=0.")



   temp=0.0 (5 runs):


NameError: name 'completion' is not defined

**Business Insight:**
- **Low Temperature (0.0-0.3):** More predictable, repeatable answers — best for factual tasks, legal, finance, compliance
- **High Temperature (0.7-1.0):** More creative, varied answers — best for brainstorming, marketing, content generation

---

### ⚠️ CRITICAL WARNING: Hallucination

LLMs are **not fact retrieval systems**. They can generate convincing but completely false information — this is called **hallucination**.

**Real Business Impact:**
- A legal summary might cite a non-existent court case
- A financial report might include made-up numbers
- A customer email might reference a policy that doesn't exist

**Rule:** Always verify important outputs. Human oversight is non-negotiable for business-critical decisions.

---
## 4. System Prompts — Research vs Legal

In [ ]:
def ask(system, user_prompt, model="gpt-4o-mini"):
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )
    return resp.choices[0].message.content.strip()

# Example 1: Research Style
research_system = "You are a professional researcher. Be factual, objective, and cite sources when possible."
print("🔬 Research Style:")
display(Markdown(ask(research_system, "What are current trends in AI adoption in banking?")))

In [ ]:
# Example 2: Legal / Compliance Style
legal_system = "You are a cautious legal advisor. Highlight risks clearly and never give definitive legal advice without proper disclaimer."
print("\n⚖️ Legal / Compliance Style:")
display(Markdown(ask(legal_system, "What are the risks of using AI to summarize customer contracts?")))

---
## 5. Same Question, Different Models + Cost Tracking

In [ ]:
import os
import litellm
from litellm import completion
from dotenv import load_dotenv

# 1. Explicitly load environment variables from your .env file
load_dotenv()

# Global cost tracker
total_cost = 0.0

def compare_models_with_cost(prompt):
    global total_cost
    
    # 2. Use explicit 'provider/' prefixes and valid model names
    for model in ["gpt-4o", "anthropic/claude-haiku-4-5"]:
        try:
            # The completion function natively pulls ANTHROPIC_API_KEY 
            # and OPENAI_API_KEY from the environment variables we loaded.
            resp = completion(
                model=model, 
                messages=[{"role": "user", "content": prompt}], 
                temperature=0.0
            )
            
            # Extract token usage safely
            try:
                prompt_tokens = resp.usage.prompt_tokens
                completion_tokens = resp.usage.completion_tokens
                total_tokens = resp.usage.total_tokens
            except AttributeError:
                prompt_tokens = completion_tokens = total_tokens = 0
            
            # Calculate cost
            try:
                cost = litellm.completion_cost(resp)
                total_cost += cost
            except Exception:
                cost = 0.0
            
            print(f"\n🔹 {model}")
            print(f"   📝 Input tokens: {prompt_tokens}")
            print(f"   📝 Output tokens: {completion_tokens}")
            print(f"   📝 Total tokens: {total_tokens}")
            print(f"   💵 This call: ${cost:.6f}")
            print(f"   💰 Running total: ${total_cost:.6f}")
            print("   " + "-" * 50)
            display(Markdown(resp.choices[0].message.content.strip()))
            
        except Exception as e:
            print(f"\n❌ Error running model {model}: {e}")

compare_models_with_cost("Summarize the benefits and risks of using generative AI in our company.")

**💡 Cost Tracking Note:** LiteLLM provides **estimated** costs based on token counts. For exact billing, check your provider dashboard (OpenAI, Anthropic, etc.).

---
## Final Takeaways

1. LLMs predict likely words, not search facts
2. **Tokens** are your real cost unit — not words
3. **Temperature** controls predictability vs creativity
4. ⚠️ **Hallucination** is real — always verify critical outputs
5. **System Prompts** are very powerful — use them to set role and tone
6. Different models give different answers at different costs
7. Always consider cost, accuracy, and risk
8. Good prompting + human oversight = best results